## 거래처 계층 구조 테이블 생성
- 오타 정제 → 입고처 제거 → Parent/Child 분류 → Duplicate 통합 → ID 재부여

In [67]:
import msoffcrypto
import io
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# 0. 파일 로드
# ─────────────────────────────────────────────
FILE_PATH = "통합170901DATA취합.xlsx"
PASSWORD  = "33"

with open(FILE_PATH, "rb") as f:
    office_file = msoffcrypto.OfficeFile(f)
    office_file.load_key(password=PASSWORD)
    decrypted = io.BytesIO()
    office_file.decrypt(decrypted)

wb = openpyxl.load_workbook(decrypted, read_only=True, data_only=True)
ws = wb["DATA"]
rows = list(ws.iter_rows(values_only=True))

columns = [
    "key", "영업담당자", "거래처명", "삼일모델명", "업체모델명", "CODE_No",
    "단가", "합의중량_g", "구단가_1",
    "구단가_2", "구단가_3", "구단가_4", "구단가_5", "구단가_6", "구단가_7",
    "대표자", "사업자번호", "주소", "업태", "종목", "납품주소",
    "마감담당자_성명", "마감담당자_이메일", "마감담당자_연락처",
    "실무담당자_성명", "실무담당자_이메일", "실무담당자_연락처",
    "운임", "결제조건_일", "제품군", "비고"
]

data = [row for row in rows[2:] if any(row)]
df_raw = pd.DataFrame(data).iloc[:, :31]
df_raw.columns = columns
df_raw = df_raw.drop(columns=["key"])
df_raw["거래처명"] = df_raw["거래처명"].astype(str).str.strip()
df_raw = df_raw[df_raw["거래처명"].notna() & (df_raw["거래처명"] != "None")].reset_index(drop=True)

print(f"원본 행 수: {len(df_raw)}")
print(f"원본 고유 거래처: {df_raw['거래처명'].nunique()}개")

원본 행 수: 5252
원본 고유 거래처: 900개


In [68]:
# ─────────────────────────────────────────────
# 1. 오타 정제
# 거래처명 전체(슬래시 앞/뒤 포함)에서 오타를 올바른 이름으로 교체
# 추가 오타 발견 시 여기에만 추가하면 됨
# ─────────────────────────────────────────────
TYPO_MAP = {
    "가지채움"      : "가치채움",
    "CJ대한퉁운"    : "CJ대한통운",
    "인펙코리아"    : "인팩코리아",
    "한국파렛트폴"  : "한국파렛트풀",
    "안팩코리아"    :  "인팩코리아",
}

def fix_typo(name: str) -> str:
    """슬래시 앞/뒤 각각 오타 수정"""
    if "/" in name:
        parts = name.split("/", 1)
        parts = [TYPO_MAP.get(p.strip(), p.strip()) for p in parts]
        return "/".join(parts)
    return TYPO_MAP.get(name, name)

before = df_raw["거래처명"].copy()
df_raw["거래처명"] = df_raw["거래처명"].apply(fix_typo)

fixed = (before != df_raw["거래처명"]).sum()
print(f"오타 수정된 행: {fixed}개")
print(df_raw.loc[before != df_raw["거래처명"], "거래처명"].value_counts().to_string())

오타 수정된 행: 15개
거래처명
한국파렛트풀/은해씨월드    4
인팩코리아/바이오플레이    4
가치채움/두더팩        2
한국파렛트풀/쉐프파트너    2
인팩코리아/고삼농협      2
인팩코리아/보보로지스     1


In [69]:
# ─────────────────────────────────────────────
# 2. 입고처 처리
# 슬래시 앞 토큰이 입고처면 → 앞을 제거하고 뒤(진짜 거래처)만 남김
# ex) 용인공장/한국이노팩  →  한국이노팩
# ─────────────────────────────────────────────
INGGO_SET = {
    "강원공장",
    "둔포공장",
    "용인공장",
    "서해공장",
    "아산공장",
    "싱싱패키지",
}

def strip_inggo(name: str) -> str:
    """입고처가 앞에 있으면 제거하고 뒤 이름만 반환"""
    if "/" in name:
        parent_token = name.split("/")[0].strip()
        if parent_token in INGGO_SET:
            return name.split("/", 1)[1].strip()
    return name

before = df_raw["거래처명"].copy()
df_raw["거래처명"] = df_raw["거래처명"].apply(strip_inggo)

stripped = (before != df_raw["거래처명"]).sum()
print(f"입고처 제거된 행: {stripped}개")
# 변경 전후 확인
changed = before[before != df_raw["거래처명"]]
for old, new in zip(changed.values, df_raw.loc[changed.index, "거래처명"].values):
    print(f"  {old}  →  {new}")

입고처 제거된 행: 213개
  강원공장/한일익스프레스  →  한일익스프레스
  둔포공장/김완수  →  김완수
  둔포공장/김완수  →  김완수
  둔포공장/미래농업연구소  →  미래농업연구소
  둔포공장/미래농업연구소  →  미래농업연구소
  둔포공장/청호수지  →  청호수지
  둔포공장/최동고  →  최동고
  둔포공장/최동고  →  최동고
  둔포공장/최동군  →  최동군
  둔포공장/최동군  →  최동군
  둔포공장/태성산업  →  태성산업
  싱싱패키지/건어맨  →  건어맨
  용인공장/새빛공조  →  새빛공조
  용인공장/우성AFC  →  우성AFC
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  용인공장/휴앤로지스  →  휴앤로지스
  용인공장/휴앤로지스  →  휴앤로지스
  용인공장/휴앤로지스  →  휴앤로지스
  용인공장/SHS  →  SHS
  용인공장/SHS  →  SHS
  용인공장/SHS  →  SHS
  용인공장/SHS  →  SHS
  용인공장/KCP이천  →  KCP이천
  용인공장/온세물류  →  온세물류
  용인공장/온세물류  →  온세물류
  용인공장/KCP이천  →  KCP이천
  용인공장/한국이노팩  →  한국이노팩
  용인공장/한국이노팩  →  한국이노팩
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨 

In [70]:
# ─────────────────────────────────────────────
# 3. 메타 정보 추출 (정제 완료된 거래처명 기준)
# ─────────────────────────────────────────────
META_COLS = [
    "거래처명", "대표자", "사업자번호", "주소", "업태", "종목", "납품주소",
    "마감담당자_성명", "마감담당자_이메일", "마감담당자_연락처",
    "실무담당자_성명", "실무담당자_이메일", "실무담당자_연락처",
    "운임", "결제조건_일"
]
df_meta = df_raw[META_COLS].drop_duplicates(subset=["거래처명"]).set_index("거래처명")
print(f"정제 후 고유 거래처: {len(df_meta)}개")

정제 후 고유 거래처: 890개


In [96]:
# ─────────────────────────────────────────────
# 4. Master 테이블 구성
# ─────────────────────────────────────────────
all_clients = set(df_raw["거래처명"].unique())
slash_clients      = {c for c in all_clients if "/" in c}
standalone_clients = {c for c in all_clients if "/" not in c}
parent_names_from_slash = {c.split("/")[0].strip() for c in slash_clients}
auto_create_parents = parent_names_from_slash - standalone_clients

print(f"전체 고유 거래처  : {len(all_clients)}개")
print(f"  슬래시(/) 패턴  : {len(slash_clients)}개")
print(f"  단독 행         : {len(standalone_clients)}개")
print(f"  자동생성 Parent : {len(auto_create_parents)}개")

records = []

# Step A: 단독 행 (고객사)
for name in sorted(standalone_clients):
    meta = df_meta.loc[name].to_dict() if name in df_meta.index else {}
    records.append({
        "거래처명" : name,
        "parent_명": None,
        "node_type": "고객사",
        **{col: meta.get(col) for col in META_COLS[1:]}
    })

# Step B: 자동 생성 Parent (슬래시에만 등장)
for name in sorted(auto_create_parents):
    records.append({
        "거래처명" : name,
        "parent_명": None,
        "node_type": "고객사",
        **{col: None for col in META_COLS[1:]}
    })

# Step C: Child (납품처)
for name in sorted(slash_clients):
    parent_name = name.split("/")[0].strip()
    child_label = name.split("/", 1)[1].strip()
    meta = df_meta.loc[name].to_dict() if name in df_meta.index else {}
    records.append({
        "거래처명" : child_label,
        "parent_명": parent_name,
        "node_type": "납품처",
        **{col: meta.get(col) for col in META_COLS[1:]}
    })

df_master = pd.DataFrame(records).reset_index(drop=True)

# ID 부여
df_master.insert(0, "거래처_ID", range(1, len(df_master) + 1))
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

# parent_id 컬럼 위치 조정
cols = df_master.columns.tolist()
cols.remove("parent_id")
cols.insert(2, "parent_id")
df_master = df_master[cols]

# 정렬 (고객사 먼저, 납품처 뒤)
type_order = {"고객사": 0, "납품처": 1}
df_master["_sort"] = df_master["node_type"].map(type_order)
df_master = df_master.sort_values(["_sort", "거래처_ID"]).drop(columns=["_sort"]).reset_index(drop=True)
df_master["거래처_ID"] = range(1, len(df_master) + 1)
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

전체 고유 거래처  : 890개
  슬래시(/) 패턴  : 252개
  단독 행         : 638개
  자동생성 Parent : 28개


In [97]:
# ─────────────────────────────────────────────
# 5. Duplicate 통합
# 같은 거래처명이 고객사 + 납품처 둘 다 있는 경우:
#   → 고객사 ID가 진짜 parent
#   → 납품처 행 삭제
#   → 납품처 ID를 가리키던 child들의 parent_id를 고객사 ID로 재연결
#
# ※ 단, 입고처(INGGO_SET)가 parent였던 케이스는 이미 Step2에서 제거됨
#   → 여기서 걸리는 건 진짜 거래처가 다른 거래처의 납품처로도 등록된 경우만
# ─────────────────────────────────────────────
고객사_names = set(df_master[df_master["node_type"] == "고객사"]["거래처명"])
납품처_names = set(df_master[df_master["node_type"] == "납품처"]["거래처명"])
duplicates = 고객사_names & 납품처_names

print(f"Duplicate 거래처: {len(duplicates)}개")
for name in sorted(duplicates):
    print(f"  • {name}")


for name in duplicates:
    # 진짜 고객사 정보 가져오기
    real_row = df_master[(df_master["거래처명"] == name) & (df_master["node_type"] == "고객사")].iloc[0]
    real_id = real_row["거래처_ID"]
    
    # 같은 이름을 가진 '납품처' 행들 찾아서 정보 업데이트
    mask_fake = (df_master["거래처명"] == name) & (df_master["node_type"] == "납품처")
    
    # 1. ID를 고객사 ID로 통일 (삭제 안 함!)
    df_master.loc[mask_fake, "거래처_ID"] = real_id
    
    # 2. 비어있는 사업자번호, 주소 등을 고객사 데이터로 채우기
    cols_to_fill = ["사업자번호", "주소", "대표자", "업태", "종목"]
    for col in cols_to_fill:
        df_master.loc[mask_fake, col] = real_row[col]


# for name in duplicates:
#     real_id = df_master.loc[
#         (df_master["거래처명"] == name) & (df_master["node_type"] == "고객사"),
#         "거래처_ID"
#     ].values[0]

#     fake_id = df_master.loc[
#         (df_master["거래처명"] == name) & (df_master["node_type"] == "납품처"),
#         "거래처_ID"
#     ].values[0]

#     # fake를 가리키던 child → real로 재연결
#     mask = df_master["parent_id"] == fake_id
#     df_master.loc[mask, "parent_id"] = real_id
#     df_master.loc[mask, "parent_명"] = name

#     # 납품처 행 삭제
#     df_master = df_master[
#         ~((df_master["거래처명"] == name) & (df_master["node_type"] == "납품처"))
#     ]

# # ID 재부여
# df_master = df_master.reset_index(drop=True)
# df_master["거래처_ID"] = range(1, len(df_master) + 1)
# name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
# df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

# print(f"\n정리 후 전체: {len(df_master)}개")
# print(f"  고객사: {(df_master['node_type']=='고객사').sum()}개")
# print(f"  납품처: {(df_master['node_type']=='납품처').sum()}개")

Duplicate 거래처: 16개
  • 건어맨
  • 남선푸드
  • 동아에스티
  • 디베랴
  • 로지스프로
  • 메이플델리
  • 명정보기술
  • 바이오플레이
  • 아이탐푸드
  • 우진수산
  • 유한산업
  • 제이월드텍
  • 지웅
  • 진원
  • 태진패키지
  • 팀프레시


In [111]:
# ─────────────────────────────────────────────
# 6. 검증
# ─────────────────────────────────────────────
print("【Parent → Child 샘플】")
for p_name in ["청호나이스", "CJ대한통운", "동원홈푸드", "한국파렛트풀", "한국이노팩", "컬리"]:
    if p_name not in name_to_id:
        print(f"  {p_name} → 없음")
        continue
    pid  = name_to_id[p_name]
    row  = df_master[df_master["거래처_ID"] == pid].iloc[0]
    kids = df_master[df_master["parent_id"] == pid][["거래처_ID","거래처명"]].values
    print(f"  [{pid}] {p_name} ({row['node_type']})")
    for kid_id, kid_name in kids[:5]:
        print(f"       └─ [{int(kid_id)}] {kid_name}")
    if len(kids) > 5:
        print(f"       └─ ... 외 {len(kids)-5}개")

# parent_id가 존재하지 않는 ID를 가리키는 행 확인 (정합성 체크)
valid_ids = set(df_master["거래처_ID"])
broken = df_master[df_master["parent_id"].notna() & ~df_master["parent_id"].isin(valid_ids)]
print(f"\n깨진 parent_id 참조: {len(broken)}개")
if len(broken) > 0:
    print(broken[["거래처_ID","거래처명","parent_id","parent_명"]])

【Parent → Child 샘플】
  [517] 청호나이스 (고객사)
       └─ [852] 미래씨엔에이치
       └─ [853] 에스엠나이스
       └─ [854] 협신산업
  [641] CJ대한통운 (고객사)
       └─ [672] 가미락
       └─ [673] 강원도옥수수총각
       └─ [674] 그림엘푸드
       └─ [675] 남천영농조합법인
       └─ [676] 도당
       └─ ... 외 21개
  [643] 동원홈푸드 (고객사)
       └─ [720] 가산공장
       └─ [721] 금천미트(대전)
       └─ [722] 대전센터
       └─ [723] 시화센터
       └─ [724] 장호원
       └─ ... 외 1개
  [664] 한국파렛트풀 (고객사)
       └─ [890] 대상
       └─ [891] 더다냉동물류
       └─ [892] 더유리빙
       └─ [893] 동원로엑스
       └─ [894] 두레냉동
       └─ ... 외 18개
  [603] 한국이노팩 (고객사)
       └─ [887] 이푸드부산
       └─ [888] 이푸드평택
       └─ [889] 현대그린푸드
  [659] 컬리 (고객사)
       └─ [855] 남양주
       └─ [856] 송파

깨진 parent_id 참조: 5개
     거래처_ID  거래처명  parent_id parent_명
866     867    덕평      773.0     팀프레시
867     868    마장      773.0     팀프레시
868     869  마장DC      773.0     팀프레시
869     870    수원      773.0     팀프레시
870     871    이천      773.0     팀프레시


In [123]:
df_master[df_master['거래처명'] =='팀프레시']

,거래처_ID,거래처명,parent_id,parent_명,node_type,대표자,사업자번호,주소,업태,종목,납품주소,마감담당자_성명,마감담당자_이메일,마감담당자_연락처,실무담당자_성명,실무담당자_이메일,실무담당자_연락처,운임,결제조건_일
558,559,팀프레시,NaN,NaN,고객사,이성일,561-88-01138,"서울특별시 송파구 위례성대로 12번길15-1,2층(방이동)","서비스,도매 및 소매업,운수업,정보서비스업","물류대행,식품,화물자동차,운송업(6톤미만)",경기도 안성시 신두만곡로 181,안희영,heeyoung.an@timf.co.kr/invoice@timf.co.kr,010-5763-3416,장덕민,duckmin.jang@timf.co.kr,010-7595-6038,None,30
772,559,팀프레시,315.0,쓰리에스테크,납품처,이성일,561-88-01138,"서울특별시 송파구 위례성대로 12번길15-1,2층(방이동)","서비스,도매 및 소매업,운수업,정보서비스업","물류대행,식품,화물자동차,운송업(6톤미만)",경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,010-5096-5361,None,None


In [122]:
df_master[df_master['parent_명'] == "팀프레시"]

,거래처_ID,거래처명,parent_id,parent_명,node_type,대표자,사업자번호,주소,업태,종목,납품주소,마감담당자_성명,마감담당자_이메일,마감담당자_연락처,실무담당자_성명,실무담당자_이메일,실무담당자_연락처,운임,결제조건_일
866,867,덕평,773.0,팀프레시,납품처,NaN,NaN,NaN,NaN,"물류대행,식품,화물자동차,운송업(6톤미만)",경기도 안성시 신두만곡로 181,NaN,NaN,NaN,이승연,NaN,010-8535-0325,None,None
867,868,마장,773.0,팀프레시,납품처,이성일,561-88-01138,"서울특별시 송파구 위례성대로 12번길15-1,2층(방이동)","서비스,도매 및 소매업,운수업,정보서비스업","물류대행,식품,화물자동차,운송업(6톤미만)",경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,NaN,None,30
868,869,마장DC,773.0,팀프레시,납품처,이성일,561-88-01138,"서울특별시 송파구 위례성대로 12번길15-1,2층(방이동)","서비스,도매 및 소매업,운수업,정보서비스업","물류대행,식품,화물자동차,운송업(6톤미만)",경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,NaN,None,30
869,870,수원,773.0,팀프레시,납품처,이성일,561-88-01138,"서울특별시 송파구 위례성대로 12번길15-1,2층(방이동)","서비스,도매 및 소매업,운수업,정보서비스업","물류대행,식품,화물자동차,운송업(6톤미만)",경기도 안성시 신두만곡로 181,박희만,heeman.park@timf.co.kr,010-2858-7977,최지원,duckmin.jang@timf.co.kr,010-4730-5355,None,30
870,871,이천,773.0,팀프레시,납품처,이성일,561-88-01138,"서울특별시 송파구 위례성대로 12번길15-1,2층(방이동)","서비스,도매 및 소매업,운수업,정보서비스업","물류대행,식품,화물자동차,운송업(6톤미만)",경기도 안성시 신두만곡로 181,박희만,heeman.park@timf.co.kr,010-2858-7977,박희만,duckmin.jang@timf.co.kr,010-2722-1525,None,30


In [29]:
# ─────────────────────────────────────────────
# 7. Detail 테이블 (FK 연결)
# ─────────────────────────────────────────────

# df_raw 거래처명도 동일하게 오타/입고처 정제 반영됨 (이미 위에서 처리)
# child는 child_label(슬래시 뒤)로 저장했으므로 거래처명 매핑 필요

# slash_clients는 원본 슬래시 포함 이름 → child_label로 변환해서 매핑
slash_label_map = {}
for name in slash_clients:
    parent_token = name.split("/")[0].strip()
    child_label  = name.split("/", 1)[1].strip()
    # 입고처였으면 df_raw에서 이미 child_label로 바뀌어있음
    slash_label_map[name] = child_label

# df_raw의 거래처명은 정제 완료 상태, name_to_id로 바로 연결
df_raw["거래처_ID"] = df_raw["거래처명"].map(name_to_id)

DETAIL_COLS = [
    "거래처_ID", "거래처명", "영업담당자",
    "삼일모델명", "업체모델명", "CODE_No",
    "단가", "합의중량_g",
    "구단가_1", "구단가_2", "구단가_3", "구단가_4", "구단가_5", "구단가_6", "구단가_7",
    "제품군", "비고"
]
df_detail = df_raw[DETAIL_COLS].reset_index(drop=True)
df_detail.insert(0, "거래내역_ID", range(1, len(df_detail) + 1))

for col in ["단가", "합의중량_g", "구단가_1", "구단가_2", "구단가_3", "구단가_4", "구단가_5", "구단가_6", "구단가_7"]:
    df_detail[col] = pd.to_numeric(df_detail[col], errors="coerce")

print(f"Detail 행 수: {len(df_detail)}")
print(f"거래처_ID 매핑 실패: {df_detail['거래처_ID'].isna().sum()}건")

Detail 행 수: 5252
거래처_ID 매핑 실패: 1073건


In [30]:
df_detail

,거래내역_ID,거래처_ID,거래처명,영업담당자,삼일모델명,업체모델명,CODE_No,단가,합의중량_g,구단가_1,구단가_2,구단가_3,구단가_4,구단가_5,구단가_6,구단가_7,제품군,비고
0,1,1.0,(유)돌코리아,최욱헌,NHD-4#2,None,None,780.0,122.0,670.00000,710.0,NaN,NaN,NaN,NaN,NaN,BOX,NaN
1,2,1.0,(유)돌코리아,최욱헌,HD4#4,None,None,780.0,122.0,670.00000,710.0,NaN,NaN,NaN,NaN,NaN,BOX,NaN
2,3,1.0,(유)돌코리아,최욱헌,JB10K,None,None,1370.0,215.0,1180.00000,1250.0,NaN,NaN,NaN,NaN,NaN,BOX,NaN
3,4,191.0,몬즈컴퍼니,정현석,SI-15K,None,None,1640.0,318.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BOX,NaN
4,5,1.0,(유)돌코리아,최욱헌,SI-5K,None,None,1340.0,191.0,1150.00000,1220.0,NaN,NaN,NaN,NaN,NaN,BOX,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5247,5248,168.0,루이스벨라,이정우,F3,None,None,3650.0,607.0,6.01318,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN
5248,5249,168.0,루이스벨라,이정우,나노박스(64배),None,None,2745.0,450.0,6.10000,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN
5249,5250,487.0,지나농원,이다영,NHD-3#3,None,None,1305.0,163.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN
5250,5251,487.0,지나농원,이다영,NHD-4#2,None,None,980.0,122.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN


In [9]:
# ─────────────────────────────────────────────
# 8. Excel 저장
# ─────────────────────────────────────────────
OUTPUT_PATH = "SQL_계층구조_거래처DB.xlsx"

COLORS = {
    "header" : "1B2A3B",
    "고객사_bg" : "D6EAF8", "고객사_font": "1A5276",
    "납품처_bg" : "D5F5E3", "납품처_font": "1E8449",
    "alt_row"  : "F4F6F7",
}

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df_master.to_excel(writer, sheet_name="Master_거래처", index=False)
    df_detail.to_excel(writer, sheet_name="Detail_거래내역", index=False)

    for sheet_name, df in [("Master_거래처", df_master), ("Detail_거래내역", df_detail)]:
        ws = writer.sheets[sheet_name]

        # 헤더
        h_fill = PatternFill("solid", start_color=COLORS["header"], end_color=COLORS["header"])
        h_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        for cell in ws[1]:
            cell.fill = h_fill
            cell.font = h_font
            cell.alignment = Alignment(horizontal="center", vertical="center")

        # 행 색상 (Master만)
        if sheet_name == "Master_거래처":
            nt_col = df_master.columns.get_loc("node_type") + 1
            for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
                nt = ws.cell(row=row_idx, column=nt_col).value
                bg   = COLORS.get(f"{nt}_bg",   "FFFFFF")
                fc   = COLORS.get(f"{nt}_font", "000000")
                bold = (nt == "고객사")
                fill = PatternFill("solid", start_color=bg, end_color=bg)
                font = Font(color=fc, bold=bold, name="Arial", size=9)
                for cell in row:
                    cell.fill = fill
                    cell.font = font
        else:
            alt = PatternFill("solid", start_color=COLORS["alt_row"], end_color=COLORS["alt_row"])
            for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
                for cell in row:
                    cell.font = Font(name="Arial", size=9)
                    if row_idx % 2 == 0:
                        cell.fill = alt

        # 열 너비
        for col in ws.columns:
            w = max((len(str(c.value)) if c.value else 0) for c in col)
            ws.column_dimensions[get_column_letter(col[0].column)].width = min(w + 4, 45)

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

print(f"✅ 저장 완료: {OUTPUT_PATH}")

PermissionError: [Errno 13] Permission denied: 'SQL_계층구조_거래처DB.xlsx'